### 0. Imports

In [ ]:
# In general all file names need to be changed and a few need to be set up, this is the rough code to put together the files I've used

In [1]:
print('Starting Marvin')
from marvin.tools import Cube
from marvin.tools.query import Query
from marvin.tools.maps import Maps

print('Running Others')
import random

import numpy as np
import json

print('Setting Marvin')
from marvin import config
config.setRelease('DR17')

folder = 'Storage/arwoods/persistent'

Starting Marvin


/home/idies/miniconda3/envs/py38/lib/python3.8/site-packages/marvin/core/exceptions.py:50: UserWarning: cannot initiate Sentry error reporting: [Errno 6] No such device or address.
  warnings.warn('cannot initiate Sentry error reporting: {0}.'.format(str(ee)),
[INFO]: No release version set. Setting default to DR17
[WARNING]: path /home/idies/workspace/sdss_sas/mangawork/manga/spectro/redux/v3_1_1/drpall-v3_1_1.fits cannot be found. Setting drpall to None. (MarvinUserWarning)
[WARNING]: path /home/idies/workspace/sdss_sas/mangawork/manga/spectro/analysis/v3_1_1/3.1.0/dapall-v3_1_1-3.1.0.fits cannot be found. Setting dapall to None. (MarvinUserWarning)


Running Others
Setting Marvin


### 1. Unpack AGNs and Make Images

In [2]:
def AGNs():
    ''' open AGN text file '''
    with open('Storage/arwoods/persistent/AGNifus.txt', 'r') as file:         # location of AGN catalogue downloaded from SDSS Website
        return file.read().split(',')
    
def bol_lum(ID):
    ''' makes bolometric flux from an image '''
    return np.sum(Cube(ID).flux.value, 0)

def make_bol_list():  # if I do this again, rewrite in steps, this takes three hours to run
    ''' adds a bolometric flux to the JSON file, for the next unincluded MaNGA ID '''
    string = ''
    with open('AGNdat.json', 'w') as file:
        res = {}
        IDs = AGNs()
        for i in range(0, len(IDs)):
            print(i)
            ID = IDs[i]
            flux = bol_lum(ID)
            res[ID] = {'image':flux.tolist()}
        json.dump(res, file)

### 2. Add Redshifts and Masses

In [3]:
def add_base_props():
    ''' adds redshift and sersic logmass to the dictionary '''
    res = {}
    with open('AGNdat1.json', 'r') as file:          # Dictionary to store first step
        res = json.load(file)
    count = 0
    for key in res:
        count += 1
        print(count)
        if not ('z' in res[key]):
            cube = Cube(key)
            if type(cube.nsa) != type(None):
                res[key]['z'] = cube.nsa['z']
                res[key]['logmass'] = cube.nsa['sersic_logmass']
            else:
                res[key]['z'] = -1
                res[key]['logmass'] = -1
                print('skipped')
    with open('AGNdat1.json', 'w') as file:
        json.dump(res, file)

### 3. Get Similar Galaxies
This step requires the sersic indicies to be added which comes from the local processing.

In [4]:
def find_matched(z, m, n):
    ''' finds galaxies similar to an AGN '''
    mmin, mmax = m-0.3, m+0.3
    nmin, nmax = n-0.5, n+0.5
    zmin, zmax = z-0.02, z+0.02
    query_filter = " nsa.z > " + str(zmin)
    res = Query(search_filter=query_filter, 
              return_params=["cube.plateifu", "cube.mangaid", "nsa.sersic_n", "nsa.z", "nsa.sersic_logmass"], 
              limit=10000).run().results
    
    donts = AGNs()
    zcheck = []
    for gal in res:  # mangaid, plateifu, sersic_n, z, sersic_logmass
        if zmin < gal[3] < zmax and not (gal[1] in donts):
            zcheck.append(gal)
    mcheck = []
    for gal in zcheck:
        if mmin < gal[4] < mmax:
            mcheck.append(gal)
    ncheck = []
    for gal in mcheck:
        if nmin < gal[2] < nmax:
            ncheck.append(gal)
            
    if len(ncheck) < 5:
        if len(mcheck) < 5:
            return [], -1      # Don't use
        return mcheck, 0       # Not Type Matched
    return ncheck, 1           # Matched

def add_matched():
    ''' should add matched IDs to each AGN entry in JSON '''
    res = {}
    with open('AGNdat1.json', 'r') as file:
        res = json.load(file)
    count = 0
    for key in res:
        count += 1
        print(count)
        sample, valid = find_matched(res[key]['z'], res[key]['logmass'], res[key]['my_n'])
        if valid != -1:
            sample = random.sample(sample, 5)
            for i in range(0, 5):
                sample[i] = (sample[i][1], sample[i][3])  # just the plate ifus
        else:
            print(key, 'failed')
            sample = []
        res[key]['matched'] = valid
        res[key]['matches'] = {}
        for pair in sample:
            res[key]['matches'][pair[0]] = {'z':pair[1]}   
    with open('AGNdat2.json', 'w') as file:                          # Store to a new dictionary with updated details
        json.dump(res, file)

### 4. Get H-alpha Maps

In [5]:
# this makes a blank file
#with open('Halpha_dict.json', 'w') as file:
#    json.dump({'filler':1}, file)

In [6]:
def halpha_dict(kill=0, minny=0, cap=50):
    ''' adds to the dict, kill is the number to skip at the start, minny is the minimum start (so as not to hit early kills), cap is how far it goes for intermediate saving '''
    killed = 0
    dat = {}
    with open('AGNdat3.json', 'r') as file:
        dat = json.load(file)
    res = {}
    with open('Halpha_dict.json', 'r') as file:
        res = json.load(file)
    if 'filler' in res:
        res.pop("filler", None)
    count = 0
    for key in dat:
        count += 1
        if minny < count < cap:
            if not key in res:
                if killed < kill:
                    print('killed', key)
                    killed += 1
                else:
                    bit = {}
                    bit['main'] = Maps(plateifu=key)['emline_gflux_ha_6564'].value.tolist()
                    bit['matches'] = {}
                    for key2 in dat[key]['matches']:
                        bit['matches'][key2] = Maps(plateifu=key2)['emline_gflux_ha_6564'].value.tolist()
                    res[key] = bit
                with open('Halpha_dict.json', 'w') as file:
                    json.dump(res, file)
                print(count, key)
                
            # normal addition till it breaks
            
#halpha_dict(kill=0, minny=340, cap=400)                     # So it can be run, skip broken pieces, and keep running. Should backup file periodically in case itb reaks

### 5. Get Match Images

In [7]:
#with open('Image_dict.json', 'w') as file:
#    json.dump({'filler':1}, file)

In [8]:
def image_dict(kill=0, minny=0, cap=50):
    ''' adds to the dict, kill is the number to skip at the start, minny is the minimum start (so as not to hit early kills), cap is how far it goes for intermediate saving '''
    killed = 0
    dat = {}
    with open('Halpha_main.json', 'r') as file:
        dat = json.load(file)
    res = {}
    with open('Image_dict.json', 'r') as file:
        res = json.load(file)
    if 'filler' in res:
        res.pop("filler", None)
    count = 0
    for key in dat:
        count += 1
        if minny < count < cap:
            if not key in res:
                if killed < kill:
                    print('killed', key)
                    killed += 1
                else:
                    bit = {}
                    # bit['main'] = bol_lum(key).tolist()  # I do already have the bolometric fluxes for the AGNs
                    bit['matches'] = {}
                    for key2 in dat[key]['matches']:
                        print('---', key2)
                        bit['matches'][key2] = bol_lum(key2).tolist()
                    res[key] = bit
                with open('Image_dict.json', 'w') as file:
                    json.dump(res, file)
                print(count, key)
                
            # normal addition till it breaks
            
#image_dict(kill=0, minny=0, cap=400)

### 6. Get H-alpha errors

In [18]:
#with open('Halpha_errs.json', 'w') as file:
#    json.dump({'filler':1}, file)

In [ ]:
def halerr_dict(kill=0, minny=0, cap=50):
    ''' adds to the dict, kill is the number to skip at the start, minny is the minimum start (so as not to hit early kills), cap is how far it goes for intermediate saving '''
    killed = 0
    dat = {}
    with open('Halpha_main.json', 'r') as file:
        dat = json.load(file)
    res = {}
    with open('Halpha_errs.json', 'r') as file:
        res = json.load(file)
    if 'filler' in res:
        res.pop("filler", None)
    count = 0
    for key in dat:
        count += 1
        if minny < count < cap:
            if not key in res:
                if killed < kill:
                    print('killed', key)
                    killed += 1
                else:
                    bit = {}
                    bit['main'] = (np.clip(1/np.array(Maps(plateifu=key)['emline_gflux_ha_6564'].ivar.tolist()), a_min=0, a_max=1e5)).tolist()  # variance, np clip is for storage
                    bit['matches'] = {}
                    for key2 in dat[key]['matches']:
                        # print('---', key2)
                        bit['matches'][key2] = (np.clip(1/np.array(Maps(plateifu=key2)['emline_gflux_ha_6564'].ivar.tolist()), a_min=0, a_max=1e5)).tolist()  # variance
                    res[key] = bit
                with open('Halpha_errs.json', 'w') as file:
                    json.dump(res, file)
                print(count, key)
                
            # normal addition till it breaks
            
halerr_dict(kill=0, minny=0, cap=400)

1 10214-3703
2 10216-9101
3 10222-12701
4 10222-6103
5 10223-3701
6 10223-3702
7 10226-6104
8 10492-6103
9 10493-1902
